# How to Use `prevent_unoptimized` Effectively in Qdrant

After a bulk upload or a configuration update, a Qdrant collection can see higher search latency. Ongoing optimizations create unindexed segments, and queries that land on those segments need a full scan to return results.

Up through Qdrant v1.17, the fix lived on the read path: `indexed_only` is a search parameter that tells Qdrant to skip unindexed regions and search only the fully optimized segments.

The problem was that points could blink: a point could appear briefly in a small segment, then disappear once that segment crossed the indexing threshold and started optimizing.

Qdrant v1.17.1 introduced a second fix, this time on the write path: the experimental `prevent_unoptimized` flag.

`prevent_unoptimized` defers the visibility of unindexed points. Once a segment starts optimizing, new points added to it stay in a deferred state until the segment finishes optimizing and becomes searchable. Qdrant still writes them to persistent storage, so you don't lose data, it just holds them back from search until they're ready.

This notebook covers how to turn on `prevent_unoptimized`, how to combine it with uploads, how to monitor optimization progress, how to measure the effect of it on query latency, and what tradeoffs the setting brings.

## Setup

Install the `qdrant-client`, `huggingface-hub` and `polars` python packages to set up a Qdrant collection, download and process the dataset for this tutorial.

We will be using a [Qdrant Cloud Free Tier Cluster]( https://qdrant.tech/documentation/cloud/create-cluster/#free-clusters.).

[Create a free cluster](https://cloud.qdrant.io/), save the associated API key and endpoint URL, and instantiate the Qdrant Client:

In [4]:
! pip install -q qdrant-client huggingface-hub polars

In [2]:
from qdrant_client import AsyncQdrantClient
from getpass import getpass

client = AsyncQdrantClient(
    url=getpass("Qdrant URL:"),
    api_key=getpass("Qdrant API key:"),
    timeout=60,
    prefer_grpc=True,
)

Qdrant URL:··········
Qdrant API key:··········


We'll be performing a few operations concurrently: searching and monitoring optimization progress. The asynchronous Qdrant client ensures these calls won't block the event loop.

We'll also be uploading a large number of embeddings in batches. For that, increase the client's timeout and prefer gRPC as the transport where possible. Both improve resilience and performance.

## Create Collections

We will create two collections, each containing 100,000 768-dimensional vectors: one with `prevent_unoptimized` set to `True`, and the other set to `False`. This will let us see the effect of `prevent_unoptimized` on query latency during the optimization phase, and on the duration of the optimization phase itself.

In [3]:
from qdrant_client import models

async def create_collection(collection_name: str, prevent_unoptimized: bool = True) -> None:
    collection_exists = await client.collection_exists(collection_name)
    if collection_exists:
        raise RuntimeError("Collection already exists, please delete it or change the name of the new collection you are trying to create")
    await client.create_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(size=768, distance=models.Distance.COSINE),
        optimizers_config=models.OptimizersConfigDiff(prevent_unoptimized=prevent_unoptimized)
    )
    print(f"Created {collection_name}")

await create_collection("prevent-unoptimized")
await create_collection("allow-unoptimized", prevent_unoptimized=False)

Created prevent-unoptimized
Created allow-unoptimized


## Download the Dataset

Next we download the [`ashraq/cohere-wiki-embedding-100k`](https://huggingface.co/datasets/ashraq/cohere-wiki-embedding-100k) dataset from HuggingFace, which contains 100,000 pre-embedded Wikipedia passages.

In [6]:
from huggingface_hub import snapshot_download

def download_dataset() -> str:
    download_path = snapshot_download(
        repo_id="ashraq/cohere-wiki-embedding-100k",
        repo_type="dataset",
        allow_patterns=["data/train-*-of-*.parquet"]
    )
    return download_path

data_path = download_dataset()

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

## Upload the Embeddings to Qdrant

Let's now load the embeddings from the Parquet files using `polars`, then batch them and upload them to our collections.

One important caveat: when uploading with `prevent_unoptimized=True`, make sure to set `wait=False`. If `wait` were set to `True`, each upload call would block until the point becomes visible to queries, i.e. until the segment it belongs to is fully optimized. This can take a relatively long time, causing timeouts and stalling the entire upload loop.

> This suggestion does not apply to the Rust and Go SDKs, or to the REST API, since they set `wait=False` by default.

In [19]:
import polars as pl

def load_data(data_path: str) -> pl.DataFrame:
    df = pl.read_parquet(source=f"{data_path.rstrip('/')}/data/train-*-of-*.parquet", columns=["emb"])
    return df

data = load_data(data_path)

In [24]:
import uuid
import asyncio

async def upload_points(collection_name: str, df: pl.DataFrame) -> None:
    for batch in df.iter_slices(1000):
        points = [
            models.PointStruct(
                id=str(uuid.uuid4()),
                vector=row["emb"]
            )
            for row in batch.iter_rows(named=True)
        ]
        await client.upsert(collection_name=collection_name, points=points, wait=False)


## Monitor Optimization Progress

You can monitor optimization progress by sending a `GET` request to the `/collections/{collection_name}/optimizations` endpoint. If you're only interested in the number of deferred points, you can get that with a `GET` request to `/collections/{collection_name}` and reading `.update_queue.deferred_points`.

The Python client provides both a `get_collection` and a `get_optimizations` method, which give us structured reports on progress.

In [25]:
import time

# We poll for optimizations every 0.5 seconds, which means
# 14_400 iterations is approximately 2hrs, which should be plenty
# of time for optimizations to complete
async def get_optimizations_progress(
    signal: asyncio.Event,
    collection_name: str,
    max_iterations: int = 14_400
) -> float:
    start = time.perf_counter()
    it = 0
    while it < max_iterations:
        optimizations: models.OptimizationsResponse
        info: models.CollectionInfo
        optimizations, info = await asyncio.gather(
            client.get_optimizations(
                collection_name=collection_name, with_="completed,queued,idle_segments"
            ),
            client.get_collection(collection_name=collection_name),
        )
        print(f"Deferred points: {info.update_queue.deferred_points or 0 if info.update_queue is not None else 0}", flush=True)
        print(f"Running optimizations: {len(optimizations.running)}", flush=True)
        print(f"Queued optimizations: {len(optimizations.queued or [])}", flush=True)
        print(f"Completed optimizations: {len(optimizations.completed or [])}", flush=True)
        if len(optimizations.running) == 0 and len(optimizations.queued or []) == 0:
            signal.set()
            break
        it += 1
        await asyncio.sleep(0.5)
    end = time.perf_counter()
    return end - start


## Send Search Queries

While the collection is being optimized, we send search queries and measure their latency. This will allow us to see the difference between a collection with `prevent_unoptimized` enabled and disabled.

As queries, we sample 1000 random vectors from our dataframe.

In [21]:
queries = data.sample(1000)["emb"].to_list()

In [26]:
async def query(
    signal: asyncio.Event,
    collection_name: str,
    queries: list[list[float]],
) -> tuple[list[float], float]:
    start = time.perf_counter()
    latencies = []
    while True:
      for query in queries:
          q_start = time.perf_counter()
          await client.query_points(collection_name=collection_name, query=query)
          q_stop = time.perf_counter()
          latencies.append(q_stop - q_start)
          print(f"Query took {q_stop - q_start}s", flush=True)
      if signal.is_set():
          break
    end = time.perf_counter()
    return latencies, end - start


## Putting Everything Together

We are now ready to upload our points, and, once the upload is done, start querying and collecting optimization status.

### Why We Poll and Query Concurrently

`query_and_optimize` runs `get_optimizations_progress` and `query` at the same time with `asyncio.gather`, rather than one after the other. This is deliberate and mirrors what actually happens in production. Searches don't pause while a collection drains its optimization backlog after a bulk load: traffic keeps coming, and it competes with indexing, merging, and vacuuming for the same CPU and I/O resources.

Running the two loops concurrently is also what lets us measure the effect we actually care about: query latency *while* the collection is under optimization pressure. `get_optimizations_progress` polls `/collections/{collection_name}/optimizations` (plus `get_collection` for `deferred_points`) every 0.5 seconds and sets the `asyncio.Event` once nothing is running or queued. The `query` loop checks that same event after each sweep through `queries` and only stops once optimizations have fully drained, so every latency sample we collect corresponds to a moment where the segments were still being worked on.

With `prevent_unoptimized=True`, watch `deferred_points` during this phase: a nonzero count is expected while segments are being optimized, since new points written to an optimizing segment are held back from search until that segment is ready. It's fine for this number to be high temporarily, as long as it drains to zero once the corresponding optimizations complete.

In [ ]:
from typing import TypedDict

class Stats(TypedDict):
    total_optimization_time: float
    total_query_time: float
    query_latencies: list[float]

async def query_and_optimize(collection_name: str, queries: list[list[float]], max_iterations: int = 14_400) -> Stats:
    results = await asyncio.gather(*[get_optimizations_progress(signal, collection_name, max_iterations), query(signal, collection_name, queries)])
    opt_time: float = results[0]
    latencies, query_time = results[1]
    return {
        "total_optimization_time": opt_time,
        "total_query_time": query_time,
        "query_latencies": latencies
    }

await asyncio.gather(upload_points("prevent-unoptimized", data), upload_points("allow-unoptimized", data))
print("Uploaded points!\n\n")
stats_prevent, stats_unopt = await asyncio.gather(query_and_optimize("prevent-unoptimized", queries), query_and_optimize("allow-unoptimized", queries))

In [29]:
import json

from statistics import mean, quantiles

class LatencyStats(TypedDict):
    min: float
    p50: float
    p95: float
    p99: float
    max: float
    mean: float
    throughput: float

def get_latency_stats(stats: Stats) -> LatencyStats:
    all_time = stats["total_query_time"]
    times = stats["query_latencies"]
    throughput = len(times) / all_time  # qps
    min_t = min(times)
    max_t = max(times)
    mean_t = mean(times)
    quant_t = quantiles(times, n=100)
    p50 = quant_t[49]
    p95 = quant_t[94]
    p99 = quant_t[98]

    return LatencyStats(
        throughput=throughput,
        min=min_t,
        max=max_t,
        mean=mean_t,
        p50=p50,
        p95=p95,
        p99=p99,
    )

print("prevent_unoptimized = true")
print(f"Total optimization time: {stats_prevent['total_optimization_time']}")
print(json.dumps(get_latency_stats(stats_prevent), indent=2))
print("\n\n")
print("prevent_unoptimized = false")
print(f"Total optimization time: {stats_unopt['total_optimization_time']}")
print(json.dumps(get_latency_stats(stats_unopt), indent=2))

prevent_unoptimized = true
Total optimization time: 0.5618014159999802
{
  "throughput": 7.161828188120137,
  "min": 0.10708374600017123,
  "max": 0.318348366000464,
  "mean": 0.1380583828190147,
  "p50": 0.11680783749989132,
  "p95": 0.18741079325009197,
  "p99": 0.2057121670003835
}



prevent_unoptimized = false
Total optimization time: 88.14651969099941
{
  "throughput": 6.997234614266481,
  "min": 0.10598464200029412,
  "max": 0.32589991400072904,
  "mean": 0.14137714988900915,
  "p50": 0.11991927650024081,
  "p95": 0.1932140347504628,
  "p99": 0.2085006406503635
}


## An Apparent Win

The better query throughput and latency, together with the faster optimization time, might suggest that `prevent_unoptimized` is always the best choice.

That's only part of the story: by definition, `prevent_unoptimized` prevents points in non-fully-optimized segments from being searched. This means searches return fewer results, if any, and are limited to the points that were uploaded first, which is also a freshness problem.

For this reason, `prevent_unoptimized` should be used with caution. If a temporary loss of results and recall is acceptable (especially in smaller collections with lower optimization times) it can improve your latency. But in bigger collections with longer optimization times, it can cause users to see partial results for a long time, until all segments are fully optimized.